In [1]:
import numpy as np
import pandas as pd
import scipy
import sklearn
import shap
import xgboost as xgb

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
print("shap:", shap.__version__)
print("xgboost:", xgb.__version__)

numpy: 1.26.4
pandas: 2.2.3
scipy: 1.14.1
sklearn: 1.5.2
shap: 0.49.1
xgboost: 2.1.4


In [2]:
import shap
import xgboost as xgb
import pandas as pd
import numpy as np
from xgboost import XGBClassifier

print("shap:", shap.__version__)
print("xgboost:", xgb.__version__)

X_small = pd.DataFrame({
    "x1": [0, 1, 0, 1],
    "x2": [1.0, 2.0, 1.5, 2.5]
})
y_small = np.array([0, 1, 0, 1])

m = XGBClassifier(
    n_estimators=5,
    max_depth=2,
    learning_rate=0.1,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    verbosity=0
)
m.fit(X_small, y_small)

explainer = shap.TreeExplainer(m)
sv = explainer.shap_values(X_small)

print("SHAP + XGBoost test OK")

shap: 0.49.1
xgboost: 2.1.4
SHAP + XGBoost test OK


In [11]:
# Imports, configurations, and feature definitions
"""
T1 interpretability script (panel-only input version)

Purpose
-------
This script implements the locked T1 interpretability stage for the thesis.
It does NOT do model selection. It simply refits the already-selected models
on 2010-2021 and computes interpretability outputs on the untouched 2022-2024
holdout.

Locked choices implemented here
-------------------------------
- Refit sample: 2010-2021
- Holdout for interpretability diagnostics: 2022-2024
- Best linear finalist: L4 (elastic-net logit winsorized branch)
- Best nonlinear finalist / overall winner: updated N3 (XGBoost no-winsor branch)
- Permutation importance metric: PR-AUC drop (average precision)
- Permutation importance repeats: 10
- Calibration plot: holdout only
- SHAP computed on full holdout
- L4 sign stability: number of 6 expanding folds with same sign as 2010-2021 refit
- FF17 reference category: Other
- FF17 Unclassified kept explicitly

Notes
-----
- SHAP values reflect contributions to the model's predictions, not causal effects.
- This script assumes Master_Modeling_Panel_v2.csv contains the locked feature matrix.
- No workbook input is required; the frozen modeling design is hard-coded below.
"""

import json
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import xgboost
from scipy.stats import spearmanr
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from xgboost import XGBClassifier

# =========================================================
# 1. USER SETTINGS (JUPYTER-FRIENDLY)
# =========================================================
BASE_DIR = Path(".")
PANEL_PATH = BASE_DIR / "Master_Modeling_Panel_v2.csv"
OUTPUT_DIR = BASE_DIR / "t1_interpretability_outputs"

# =========================================================
# 2. LOCKED GLOBAL SETTINGS
# =========================================================
TRAIN_END_YEAR = 2021
TEST_START_YEAR = 2022
TEST_END_YEAR = 2024
TARGET_COL = "targeted_in_year"
CATEGORICAL_FEATURE = "ff17_for_model"
ID_COLS = ["permno", "year"]
SEED = 42

PERMUTATION_REPEATS = 10
CALIBRATION_BINS = 10
TOP_K_LINEAR_MAIN = 12
TOP_K_COMPARISON = 10
TOP_K_PERM_IMPORTANCE_PLOT = 20
TOP_K_SHAP_PLOT = 20

# Locked L4 finalist
L4_PARAMS = {
    "penalty": "elasticnet",
    "solver": "saga",
    "C": 0.1,
    "l1_ratio": 0.5,
    "max_iter": 20000,
    "random_state": SEED,
}

# Locked updated N3 finalist / overall winner
N3_PARAMS = {
    "n_estimators": 800,
    "max_depth": 3,
    "learning_rate": 0.01,
    "min_child_weight": 10,
    "subsample": 0.6,
    "colsample_bytree": 0.5,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": SEED,
    "n_jobs": -1,
    "verbosity": 0,
}

# =========================================================
# 3. LOCKED RAW PREDICTOR SET
#    These are frozen directly from the final modeling panel design.
# =========================================================
continuous_features = [
    "roe",
    "assets_to_equity",
    "current_ratio",
    "ebitda",
    "ebitda_margin",
    "sales_to_assets",
    "sales_growth",
    "interest_coverage",
    "net_debt_to_ebitda",
    "fcf_to_capex",
    "market_cap",
    "ret_3m",
    "ret_6m",
    "ret_1y",
    "ret_2y",
    "ret_3y",
    "ret_5y",
    "volatility_30d",
    "volatility_90d",
    "volatility_180d",
    "turnover_30d",
    "dividend_payout_ratio",
    "buyback_yield",
    "price_to_book",
    "ev_to_sales",
    "ev_to_ebitda",
    "tobins_q",
    "fcf_yield",
    "prior_campaign_count_3y",
    "prior_settlement_count_3y",
    "prior_management_favorable_count_3y",
    "prior_dissident_favorable_count_3y",
    "prior_campaign_count_5y",
    "prior_settlement_count_5y",
    "prior_management_favorable_count_5y",
    "prior_dissident_favorable_count_5y",
    "ff49_other_permno_years_in_category",
    "ret_3m_rel_peer",
    "ret_6m_rel_peer",
    "ret_1y_rel_peer",
    "ret_2y_rel_peer",
    "ret_3y_rel_peer",
    "ret_5y_rel_peer",
    "log_ev_to_sales_rel_peer",
    "log_price_to_book_rel_peer",
    "log_tobins_q_rel_peer",
    "log_ev_to_ebitda_rel_peer",
    "ebitda_margin_rel_peer",
    "sales_growth_rel_peer",
    "roe_rel_peer",
    "board_size",
    "board_female_ratio",
    "ceo_tenure",
    "top10_inst_conc_within_13f",
    "inst_ownership_pct_13f",
]

binary_features = [
    "prior_activism_3y",
    "prior_activism_5y",
    "history_3y_observed",
    "history_5y_observed",
    "ff49_unclassified",
    "classified_board",
    "poison_pill",
    "dual_class",
    "iss_match_found",
    "rm_stale_gt_730",
    "board_match_found",
    "board_stale_gt_365",
    "ceo_is_female",
    "ceo_chair_duality",
    "ceo_match_found",
    "ceo_stale_gt_365",
    "inst_match_found",
]

all_predictors = continuous_features + binary_features + [CATEGORICAL_FEATURE]

# =========================================================
# 4. FROZEN FEATURE METADATA
# =========================================================
HELPER_FEATURES = {
    "history_3y_observed",
    "history_5y_observed",
    "iss_match_found",
    "rm_stale_gt_730",
    "board_match_found",
    "board_stale_gt_365",
    "ceo_match_found",
    "ceo_stale_gt_365",
    "inst_match_found",
    "ff49_unclassified",
    "ff49_other_permno_years_in_category",
}

BLOCK_MAP = {
    # Core accounting / operating
    "roe": "Accounting / operating",
    "assets_to_equity": "Accounting / operating",
    "current_ratio": "Accounting / operating",
    "ebitda": "Accounting / operating",
    "ebitda_margin": "Accounting / operating",
    "sales_to_assets": "Accounting / operating",
    "sales_growth": "Accounting / operating",
    "interest_coverage": "Accounting / operating",
    "net_debt_to_ebitda": "Accounting / operating",
    "fcf_to_capex": "Accounting / operating",
    # Market / returns / volatility / liquidity
    "market_cap": "Market",
    "ret_3m": "Market",
    "ret_6m": "Market",
    "ret_1y": "Market",
    "ret_2y": "Market",
    "ret_3y": "Market",
    "ret_5y": "Market",
    "volatility_30d": "Market",
    "volatility_90d": "Market",
    "volatility_180d": "Market",
    "turnover_30d": "Market",
    # Valuation / payout
    "dividend_payout_ratio": "Valuation / payout",
    "buyback_yield": "Valuation / payout",
    "price_to_book": "Valuation / payout",
    "ev_to_sales": "Valuation / payout",
    "ev_to_ebitda": "Valuation / payout",
    "tobins_q": "Valuation / payout",
    "fcf_yield": "Valuation / payout",
    # Activism history
    "prior_activism_3y": "Activism history",
    "prior_activism_5y": "Activism history",
    "prior_campaign_count_3y": "Activism history",
    "prior_settlement_count_3y": "Activism history",
    "prior_management_favorable_count_3y": "Activism history",
    "prior_dissident_favorable_count_3y": "Activism history",
    "prior_campaign_count_5y": "Activism history",
    "prior_settlement_count_5y": "Activism history",
    "prior_management_favorable_count_5y": "Activism history",
    "prior_dissident_favorable_count_5y": "Activism history",
    "history_3y_observed": "Activism history",
    "history_5y_observed": "Activism history",
    # Peer-relative block
    "ff49_other_permno_years_in_category": "Peer-relative / industry quality",
    "ff49_unclassified": "Peer-relative / industry quality",
    "ret_3m_rel_peer": "Peer-relative",
    "ret_6m_rel_peer": "Peer-relative",
    "ret_1y_rel_peer": "Peer-relative",
    "ret_2y_rel_peer": "Peer-relative",
    "ret_3y_rel_peer": "Peer-relative",
    "ret_5y_rel_peer": "Peer-relative",
    "log_ev_to_sales_rel_peer": "Peer-relative",
    "log_price_to_book_rel_peer": "Peer-relative",
    "log_tobins_q_rel_peer": "Peer-relative",
    "log_ev_to_ebitda_rel_peer": "Peer-relative",
    "ebitda_margin_rel_peer": "Peer-relative",
    "sales_growth_rel_peer": "Peer-relative",
    "roe_rel_peer": "Peer-relative",
    # Governance
    "classified_board": "Governance",
    "poison_pill": "Governance",
    "dual_class": "Governance",
    "iss_match_found": "Governance",
    "rm_stale_gt_730": "Governance",
    "board_size": "Governance",
    "board_female_ratio": "Governance",
    "board_match_found": "Governance",
    "board_stale_gt_365": "Governance",
    "ceo_tenure": "Governance",
    "ceo_is_female": "Governance",
    "ceo_chair_duality": "Governance",
    "ceo_match_found": "Governance",
    "ceo_stale_gt_365": "Governance",
    # Ownership
    "top10_inst_conc_within_13f": "Ownership",
    "inst_ownership_pct_13f": "Ownership",
    "inst_match_found": "Ownership",
    # Categorical control
    CATEGORICAL_FEATURE: "FF17 industry control",
}

ROLE_MAP = {
    feat: ("helper / quality indicator" if feat in HELPER_FEATURES else "substantive predictor")
    for feat in continuous_features + binary_features
}
ROLE_MAP[CATEGORICAL_FEATURE] = "industry control"

RAW_TYPE_MAP = {feat: "continuous" for feat in continuous_features}
RAW_TYPE_MAP.update({feat: "binary" for feat in binary_features})
RAW_TYPE_MAP[CATEGORICAL_FEATURE] = "categorical"

FF17_CATEGORIES = [
    "Cars",
    "Chems",
    "Clths",
    "Cnstr",
    "Cnsum",
    "Durbl",
    "FabPr",
    "Finan",
    "Food",
    "Machn",
    "Mines",
    "Oil",
    "Other",
    "Rtail",
    "Steel",
    "Trans",
    "Utils",
    "Unclassified",
]

FF17_NON_REFERENCE = [c for c in FF17_CATEGORIES if c != "Other"]
FF17_DUMMY_NAME_MAP = {c: f"ff17_{c}" for c in FF17_NON_REFERENCE}

In [12]:
# Preprocessing and model fitting helpers
# =========================================================
# 5. HELPERS
# =========================================================
def evaluate_predictions(y_true: np.ndarray, y_prob: np.ndarray, model_name: str) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    return {
        "model": model_name,
        "pr_auc": average_precision_score(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "brier_score": brier_score_loss(y_true, y_prob),
        "mean_predicted_probability": float(np.mean(y_prob)),
        "actual_positive_rate": float(np.mean(y_true)),
        "rows": int(len(y_true)),
        "positives": int(np.sum(y_true)),
    }


def build_transformed_metadata() -> pd.DataFrame:
    rows = []
    for feat in continuous_features + binary_features:
        rows.append(
            {
                "feature_name": feat,
                "predictor": feat,
                "block": BLOCK_MAP.get(feat, "Unknown"),
                "role_in_matrix": ROLE_MAP.get(feat, "substantive predictor"),
                "raw_type": RAW_TYPE_MAP.get(feat, "unknown"),
                "feature_family": "binary" if feat in binary_features else "numeric",
                "is_helper_or_quality": feat in HELPER_FEATURES,
                "is_ff17_dummy": False,
                "display_name": feat,
            }
        )

    for cat in FF17_NON_REFERENCE:
        rows.append(
            {
                "feature_name": FF17_DUMMY_NAME_MAP[cat],
                "predictor": CATEGORICAL_FEATURE,
                "block": "FF17 industry control",
                "role_in_matrix": "industry control",
                "raw_type": "derived FF17 dummy",
                "feature_family": "ff17_dummy",
                "is_helper_or_quality": False,
                "is_ff17_dummy": True,
                "display_name": f"FF17: {cat}",
            }
        )

    return pd.DataFrame(rows)


def check_versions(output_dir: Path) -> pd.DataFrame:
    versions = pd.DataFrame(
        {
            "package": ["shap", "xgboost"],
            "version": [shap.__version__, xgboost.__version__],
        }
    )
    versions.to_csv(output_dir / "package_versions.csv", index=False)
    print("\nPackage versions:")
    print(versions.to_string(index=False))
    print()
    return versions

def load_panel() -> pd.DataFrame:
    df = pd.read_csv(PANEL_PATH, low_memory=False)

    # Only check raw columns that should already exist in the CSV
    raw_required_cols = (
        continuous_features
        + binary_features
        + [TARGET_COL]
        + ID_COLS
        + ["ff17_short"]
    )

    missing_required = [c for c in raw_required_cols if c not in df.columns]
    if missing_required:
        raise ValueError(f"Missing required raw columns in panel: {missing_required}")

    # Build the model categorical feature from the raw FF17 label
    df[CATEGORICAL_FEATURE] = df["ff17_short"].fillna("Unclassified").astype(str)

    # Keep FF17 values inside the locked category universe
    df[CATEGORICAL_FEATURE] = df[CATEGORICAL_FEATURE].where(
        df[CATEGORICAL_FEATURE].isin(FF17_CATEGORIES),
        "Unclassified"
    )

    # Restrict to the frozen sample window
    df = df[df["year"].between(2010, 2024)].copy()

    if df[TARGET_COL].isna().any():
        raise ValueError(f"{TARGET_COL} contains missing values; fix the panel before running T1.")

    # Final check after derived feature construction
    final_required_cols = all_predictors + [TARGET_COL] + ID_COLS
    missing_final = [c for c in final_required_cols if c not in df.columns]
    if missing_final:
        raise ValueError(f"Missing required columns after FF17 construction: {missing_final}")

    return df
    

def split_panel(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_df = df[df["year"].between(2010, TRAIN_END_YEAR)].copy()
    test_df = df[df["year"].between(TEST_START_YEAR, TEST_END_YEAR)].copy()

    if train_df.empty or test_df.empty:
        raise ValueError("Train or test split is empty. Check the year filtering.")

    return train_df, test_df


@dataclass
class ManualPreprocessor:
    """
    Manual preprocessing to mirror the frozen thesis setup:
    - continuous numeric: median impute
    - winsorization only in winsorized branches
    - scaling only in linear branches
    - binary indicators: missing -> 0
    - FF17 categorical: one-hot encode with Other omitted as reference, Unclassified explicit
    """

    numeric_cols: list[str]
    binary_cols: list[str]
    ff17_col: str = CATEGORICAL_FEATURE

    medians_: dict[str, float] = field(default_factory=dict)
    lower_: dict[str, float] = field(default_factory=dict)
    upper_: dict[str, float] = field(default_factory=dict)
    means_: dict[str, float] = field(default_factory=dict)
    stds_: dict[str, float] = field(default_factory=dict)
    feature_names_: list[str] = field(default_factory=list)

    def fit(self, X: pd.DataFrame, winsorize: bool, scale: bool) -> "ManualPreprocessor":
        self.medians_ = {
            c: float(pd.to_numeric(X[c], errors="coerce").median(skipna=True))
            for c in self.numeric_cols
        }

        if winsorize:
            self.lower_ = {
                c: float(pd.to_numeric(X[c], errors="coerce").quantile(0.01))
                for c in self.numeric_cols
            }
            self.upper_ = {
                c: float(pd.to_numeric(X[c], errors="coerce").quantile(0.99))
                for c in self.numeric_cols
            }
        else:
            self.lower_ = {}
            self.upper_ = {}

        transformed_no_scale = self.transform(X, winsorize=winsorize, scale=False)

        if scale:
            self.means_ = {c: float(transformed_no_scale[c].mean()) for c in self.numeric_cols}
            self.stds_ = {}
            for c in self.numeric_cols:
                std = float(transformed_no_scale[c].std(ddof=0))
                self.stds_[c] = std if std > 0 else 1.0
        else:
            self.means_ = {}
            self.stds_ = {}

        transformed_final = self.transform(X, winsorize=winsorize, scale=scale)
        self.feature_names_ = list(transformed_final.columns)
        return self

    def _encode_ff17(self, series: pd.Series) -> pd.DataFrame:
        vals = series.fillna("Unclassified").astype(str)
        out = pd.DataFrame(index=series.index)
        for cat in FF17_NON_REFERENCE:
            out[FF17_DUMMY_NAME_MAP[cat]] = (vals == cat).astype(float)
        return out

    def transform(self, X: pd.DataFrame, winsorize: bool, scale: bool) -> pd.DataFrame:
        num = X[self.numeric_cols].copy()
        for c in self.numeric_cols:
            num[c] = pd.to_numeric(num[c], errors="coerce").fillna(self.medians_[c])

        if winsorize:
            for c in self.numeric_cols:
                num[c] = num[c].clip(self.lower_[c], self.upper_[c])

        if scale:
            for c in self.numeric_cols:
                num[c] = (num[c] - self.means_[c]) / self.stds_[c]

        binary_df = X[self.binary_cols].copy()
        for c in self.binary_cols:
            binary_df[c] = pd.to_numeric(binary_df[c], errors="coerce").fillna(0).astype(float)

        ff17_df = self._encode_ff17(X[self.ff17_col])
        out = pd.concat([num, binary_df, ff17_df], axis=1)
        out = out.astype(np.float64)
        return out


def feature_standard_deviations(X_train_transformed: pd.DataFrame) -> pd.Series:
    return X_train_transformed.std(ddof=0)


# =========================================================
# 6. MODEL FIT HELPERS
# =========================================================
def fit_l4(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> tuple[LogisticRegression, ManualPreprocessor, pd.DataFrame, pd.DataFrame, np.ndarray]:
    preprocessor = ManualPreprocessor(
        numeric_cols=continuous_features,
        binary_cols=binary_features,
    )
    preprocessor.fit(train_df[all_predictors], winsorize=True, scale=True)

    X_train = preprocessor.transform(train_df[all_predictors], winsorize=True, scale=True)
    X_test = preprocessor.transform(test_df[all_predictors], winsorize=True, scale=True)

    y_train = train_df[TARGET_COL].astype(int).to_numpy()

    model = LogisticRegression(**L4_PARAMS)
    model.fit(X_train, y_train)

    probs = model.predict_proba(X_test)[:, 1]
    return model, preprocessor, X_train, X_test, probs


def fit_n3(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> tuple[XGBClassifier, ManualPreprocessor, pd.DataFrame, pd.DataFrame, np.ndarray]:
    preprocessor = ManualPreprocessor(
        numeric_cols=continuous_features,
        binary_cols=binary_features,
    )
    preprocessor.fit(train_df[all_predictors], winsorize=False, scale=False)

    X_train = preprocessor.transform(train_df[all_predictors], winsorize=False, scale=False)
    X_test = preprocessor.transform(test_df[all_predictors], winsorize=False, scale=False)

    y_train = train_df[TARGET_COL].astype(int).to_numpy()

    model = XGBClassifier(**N3_PARAMS)
    model.fit(X_train, y_train)

    probs = model.predict_proba(X_test)[:, 1]
    return model, preprocessor, X_train, X_test, probs

In [13]:
# L4 outputs (coefficients, odds-ratios, sign stability)
# =========================================================
# 7. L4 OUTPUTS
# =========================================================
def compute_l4_sign_stability(
    full_model: LogisticRegression,
    train_df_2010_2021: pd.DataFrame,
) -> pd.Series:
    """
    Sign stability = number of six expanding-window folds (validating 2016...2021)
    whose coefficient sign matches the sign in the full 2010-2021 refit.
    """
    final_sign = np.sign(full_model.coef_.ravel())
    feature_names = None
    stable_counts = None

    fold_years = [2016, 2017, 2018, 2019, 2020, 2021]

    for fold_year in fold_years:
        fold_train = train_df_2010_2021[train_df_2010_2021["year"] < fold_year].copy()

        preprocessor = ManualPreprocessor(
            numeric_cols=continuous_features,
            binary_cols=binary_features,
        )
        preprocessor.fit(fold_train[all_predictors], winsorize=True, scale=True)
        X_fold_train = preprocessor.transform(fold_train[all_predictors], winsorize=True, scale=True)

        model = LogisticRegression(**L4_PARAMS)
        model.fit(X_fold_train, fold_train[TARGET_COL].astype(int).to_numpy())

        coef_sign = np.sign(model.coef_.ravel())

        if feature_names is None:
            feature_names = list(X_fold_train.columns)
            stable_counts = np.zeros(len(feature_names), dtype=int)

        stable_counts += (coef_sign == final_sign).astype(int)

    return pd.Series(stable_counts, index=feature_names, name="sign_matches_out_of_6")


def build_l4_tables(
    model: LogisticRegression,
    X_train_transformed: pd.DataFrame,
    transformed_meta: pd.DataFrame,
    sign_stability: pd.Series,
    output_dir: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    feature_names = list(X_train_transformed.columns)
    coef = model.coef_.ravel()
    stds = feature_standard_deviations(X_train_transformed).reindex(feature_names)

    table = pd.DataFrame(
        {
            "feature_name": feature_names,
            "coefficient": coef,
            "odds_ratio": np.exp(coef),
            "training_feature_sd": stds.values,
            "standardized_effect": coef * stds.values,
            "sign_stability": [f"{int(sign_stability.loc[f])}/6" for f in feature_names],
        }
    )

    table = table.merge(transformed_meta, on="feature_name", how="left")
    table["absolute_standardized_effect"] = table["standardized_effect"].abs()
    table["absolute_coefficient"] = table["coefficient"].abs()
    table = table.sort_values("absolute_standardized_effect", ascending=False).reset_index(drop=True)
    table.to_csv(output_dir / "l4_full_appendix_coefficients.csv", index=False)

    main_cols = [
        "feature_name",
        "display_name",
        "block",
        "role_in_matrix",
        "coefficient",
        "odds_ratio",
        "standardized_effect",
        "sign_stability",
    ]
    main_table = table.loc[:, main_cols].head(TOP_K_LINEAR_MAIN).copy()
    main_table.to_csv(output_dir / "l4_main_text_top_features.csv", index=False)

    return table, main_table

In [24]:
# N3 outputs (permutation importance and SHAP)
# =========================================================
# 8. N3 PERMUTATION IMPORTANCE
# =========================================================
def permutation_importance_pr_auc(
    model: XGBClassifier,
    X_test: pd.DataFrame,
    y_test: np.ndarray,
    repeats: int,
    seed: int,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)

    baseline_probs = model.predict_proba(X_test)[:, 1]
    baseline_ap = average_precision_score(y_test, baseline_probs)

    results = []
    X_values = X_test.to_numpy(copy=True)
    n_rows = X_values.shape[0]
    columns = list(X_test.columns)

    for j, col in enumerate(columns):
        original_col = X_values[:, j].copy()
        drops = []

        for _ in range(repeats):
            perm = rng.permutation(n_rows)
            X_values[:, j] = original_col[perm]
            shuffled_probs = model.predict_proba(X_values)[:, 1]
            shuffled_ap = average_precision_score(y_test, shuffled_probs)
            drops.append(baseline_ap - shuffled_ap)

        X_values[:, j] = original_col

        results.append(
            {
                "feature_name": col,
                "baseline_pr_auc": baseline_ap,
                "importance_mean_pr_auc_drop": float(np.mean(drops)),
                "importance_std_pr_auc_drop": float(np.std(drops, ddof=0)),
            }
        )

    out = pd.DataFrame(results).sort_values(
        "importance_mean_pr_auc_drop",
        ascending=False,
    ).reset_index(drop=True)
    return out


def plot_permutation_importance(table: pd.DataFrame, output_dir: Path) -> None:
    plot_df = table.head(TOP_K_PERM_IMPORTANCE_PLOT).copy()
    plot_df = plot_df.sort_values("importance_mean_pr_auc_drop", ascending=True)

    plt.figure(figsize=(10, 8))
    plt.barh(
        plot_df["display_name"],
        plot_df["importance_mean_pr_auc_drop"],
        xerr=plot_df["importance_std_pr_auc_drop"],
    )
    plt.xlabel("Mean PR-AUC drop after permutation")
    plt.ylabel("")
    plt.title("N3 Permutation Importance on 2022–2024 Holdout")
    plt.tight_layout()
    plt.savefig(output_dir / "n3_permutation_importance_top20.png", dpi=220, bbox_inches="tight")
    plt.close()


# =========================================================
# 9. SHAP COMPATIBILITY HELPERS
# =========================================================
def compute_shap_explanation(model: XGBClassifier, X_test: pd.DataFrame) -> Any:
    """
    Compute SHAP TreeExplainer values for the XGBoost model.
    The source-file patch above handles the XGBoost 3.x bracket bug.
    We pass numpy float64 and normalise output shapes.
    """
    X_np = X_test.to_numpy(dtype=np.float64)
    feature_names = list(X_test.columns)

    explainer = shap.TreeExplainer(model)

    # -- try the modern __call__ API first --
    try:
        shap_exp = explainer(X_np)
        if hasattr(shap_exp, "values"):
            vals = np.asarray(shap_exp.values)
            base = np.asarray(shap_exp.base_values)
            if vals.ndim == 3:
                vals = vals[:, :, 1]
            if base.ndim == 2:
                base = base[:, 1]
            if base.ndim == 0:
                base = np.repeat(float(base), X_np.shape[0])
            return shap.Explanation(
                values=vals,
                base_values=base,
                data=X_np,
                feature_names=feature_names,
            )
    except Exception:
        pass

    # -- fallback to legacy shap_values() --
    shap_values = explainer.shap_values(X_np)
    expected_value = explainer.expected_value

    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    shap_values = np.asarray(shap_values, dtype=np.float64)

    base_values = np.asarray(expected_value)
    if base_values.ndim == 0:
        base_values = np.repeat(float(base_values), X_np.shape[0])
    elif base_values.ndim >= 1:
        base_values = base_values.ravel()
        if len(base_values) == 1:
            base_values = np.repeat(base_values[0], X_np.shape[0])
        elif len(base_values) == 2:
            base_values = np.repeat(base_values[1], X_np.shape[0])

    return shap.Explanation(
        values=shap_values,
        base_values=base_values,
        data=X_np,
        feature_names=feature_names,
    )


def get_shap_values_matrix(shap_obj: Any) -> np.ndarray:
    if hasattr(shap_obj, "values"):
        return np.asarray(shap_obj.values)
    return np.asarray(shap_obj["values"])


def get_shap_base_values(shap_obj: Any, n_rows: int) -> np.ndarray:
    if hasattr(shap_obj, "base_values"):
        base_values = np.asarray(shap_obj.base_values)
    else:
        base_values = np.asarray(shap_obj["base_values"])

    if base_values.ndim == 0:
        return np.repeat(base_values, n_rows)
    if len(base_values) == 1:
        return np.repeat(base_values[0], n_rows)
    return base_values


# =========================================================
# 10. N3 SHAP OUTPUTS
# =========================================================
def compute_shap_outputs(
    model: XGBClassifier,
    X_test: pd.DataFrame,
    test_df: pd.DataFrame,
    probs: np.ndarray,
    output_dir: Path,
) -> Any:
    shap_obj = compute_shap_explanation(model, X_test)
    shap_values = get_shap_values_matrix(shap_obj)
    base_values = get_shap_base_values(shap_obj, n_rows=X_test.shape[0])

    np.savez_compressed(
        output_dir / "n3_shap_values_holdout.npz",
        shap_values=shap_values,
        base_values=base_values,
        feature_names=np.asarray(list(X_test.columns), dtype=object),
        predicted_probabilities=probs,
        permno=test_df["permno"].to_numpy(),
        year=test_df["year"].to_numpy(),
        y_true=test_df[TARGET_COL].astype(int).to_numpy(),
    )

    return shap_obj


def build_shap_tables(
    shap_obj: Any,
    X_test: pd.DataFrame,
    transformed_meta: pd.DataFrame,
    output_dir: Path,
) -> pd.DataFrame:
    shap_values = get_shap_values_matrix(shap_obj)

    shap_table = pd.DataFrame(
        {
            "feature_name": X_test.columns,
            "mean_abs_shap": np.abs(shap_values).mean(axis=0),
        }
    )
    shap_table = shap_table.merge(transformed_meta, on="feature_name", how="left")
    shap_table = shap_table.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    shap_table.to_csv(output_dir / "n3_shap_mean_abs_summary.csv", index=False)
    return shap_table


def plot_shap_summary(shap_obj: Any, X_test: pd.DataFrame, output_dir: Path) -> None:
    shap_values = get_shap_values_matrix(shap_obj)
    feature_names = list(X_test.columns)
    X_np = X_test.to_numpy(dtype=np.float64)

    plt.figure()
    shap.summary_plot(shap_values, X_np, feature_names=feature_names, show=False, max_display=TOP_K_SHAP_PLOT)
    plt.tight_layout()
    plt.savefig(output_dir / "n3_shap_beeswarm_top20.png", dpi=220, bbox_inches="tight")
    plt.close()

    plt.figure()
    shap.summary_plot(shap_values, X_np, feature_names=feature_names, plot_type="bar", show=False, max_display=TOP_K_SHAP_PLOT)
    plt.tight_layout()
    plt.savefig(output_dir / "n3_shap_bar_top20.png", dpi=220, bbox_inches="tight")
    plt.close()


def choose_dependence_features(shap_table: pd.DataFrame, output_dir: Path) -> tuple[list[str], pd.DataFrame]:
    """
    Locked dependence-plot rule:
    - start from mean absolute SHAP ranking
    - exclude helper/coverage indicators
    - exclude FF17 dummies
    - enforce block diversity where possible
    - select exactly 3 features
    """
    candidates = shap_table.loc[
        (~shap_table["is_helper_or_quality"].fillna(False))
        & (~shap_table["is_ff17_dummy"].fillna(False))
    ].copy()

    selected = []
    selected_rows = []
    seen_blocks = set()

    for _, row in candidates.iterrows():
        feature = row["feature_name"]
        block = row["block"]

        if not selected:
            selected.append(feature)
            selected_rows.append(row)
            seen_blocks.add(block)
        elif block not in seen_blocks:
            selected.append(feature)
            selected_rows.append(row)
            seen_blocks.add(block)

        if len(selected) == 3:
            break

    if len(selected) < 3:
        for _, row in candidates.iterrows():
            feature = row["feature_name"]
            if feature not in selected:
                selected.append(feature)
                selected_rows.append(row)
            if len(selected) == 3:
                break

    selected_df = pd.DataFrame(selected_rows)
    selected_df.to_csv(output_dir / "n3_selected_dependence_features.csv", index=False)
    return selected, selected_df


def plot_shap_dependence(
    shap_obj: Any,
    X_test: pd.DataFrame,
    selected_features: list[str],
    output_dir: Path,
) -> None:
    shap_values = get_shap_values_matrix(shap_obj)
    feature_names = list(X_test.columns)

    for feature in selected_features:
        feat_idx = feature_names.index(feature)

        plot_df = pd.DataFrame(
            {
                "feature_value": pd.Series(X_test[feature], dtype="float64"),
                "shap_value": pd.Series(shap_values[:, feat_idx], dtype="float64"),
            }
        ).replace([np.inf, -np.inf], np.nan).dropna()

        display_note = ""

        if feature == "market_cap":
            plot_df = plot_df.loc[plot_df["feature_value"] > 0].copy()
            display_note = "x-axis in log scale"

        elif feature == "current_ratio":
            upper = plot_df["feature_value"].quantile(0.99)
            plot_df = plot_df.loc[plot_df["feature_value"] <= upper].copy()
            display_note = "display trimmed at 99th percentile"

        fig, ax = plt.subplots(figsize=(8, 6))

        ax.scatter(
            plot_df["feature_value"],
            plot_df["shap_value"],
            s=20,
            alpha=0.6,
        )

        if feature == "market_cap":
            ax.set_xscale("log")

        ax.set_xlabel(feature)
        ax.set_ylabel(f"SHAP value for\n{feature}")

        title = f"N3 SHAP dependence: {feature}"
        if display_note:
            title = f"{title}\n{display_note}"

        ax.set_title(title)

        plt.tight_layout()
        plt.savefig(output_dir / f"n3_dependence_{feature}.png", dpi=220, bbox_inches="tight")
        plt.close()

In [25]:
df = load_panel()
train_df, test_df = split_panel(df)

n3_model, _, _, X_test_n3, n3_probs = fit_n3(train_df, test_df)

shap_obj = compute_shap_outputs(
    model=n3_model,
    X_test=X_test_n3,
    test_df=test_df,
    probs=n3_probs,
    output_dir=OUTPUT_DIR,
)

selected_features = ["prior_campaign_count_5y", "market_cap", "log_tobins_q_rel_peer"]

plot_shap_dependence(
    shap_obj=shap_obj,
    X_test=X_test_n3,
    selected_features=selected_features,
    output_dir=OUTPUT_DIR,
)

print("Dependence plots regenerated.")

Dependence plots regenerated.


In [15]:
# Calibration
# =========================================================
# 11. CALIBRATION
# =========================================================
def build_calibration_outputs(y_test: np.ndarray, probs: np.ndarray, output_dir: Path) -> pd.DataFrame:
    frac_pos, mean_pred = calibration_curve(
        y_test,
        probs,
        n_bins=CALIBRATION_BINS,
        strategy="quantile",
    )

    quantiles = np.linspace(0, 1, CALIBRATION_BINS + 1)
    edges = np.quantile(probs, quantiles)

    rows = []
    for i in range(CALIBRATION_BINS):
        left = edges[i]
        right = edges[i + 1]

        if i < CALIBRATION_BINS - 1:
            mask = (probs >= left) & (probs < right)
        else:
            mask = (probs >= left) & (probs <= right)

        if mask.sum() == 0:
            continue

        rows.append(
            {
                "bin": i + 1,
                "count": int(mask.sum()),
                "mean_predicted_probability": float(np.mean(probs[mask])),
                "observed_positive_rate": float(np.mean(y_test[mask])),
            }
        )

    calib_table = pd.DataFrame(rows)
    calib_table.to_csv(output_dir / "n3_calibration_bins.csv", index=False)

    max_axis = float(max(np.max(mean_pred), np.max(frac_pos), 1e-8))

    plt.figure(figsize=(7, 6))
    plt.plot([0, max_axis], [0, max_axis], linestyle="--")
    plt.plot(mean_pred, frac_pos, marker="o")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed positive rate")
    plt.title("N3 Calibration on 2022–2024 Holdout")
    plt.tight_layout()
    plt.savefig(output_dir / "n3_calibration_plot.png", dpi=220, bbox_inches="tight")
    plt.close()

    return calib_table

In [16]:
# L4 vs N3 Comparison
# =========================================================
# 12. L4 VS N3 COMPARISON HELPERS
# =========================================================
def approximate_n3_direction(feature_values: np.ndarray, shap_values: np.ndarray) -> tuple[str, float]:
    """
    Broad directional association for comparison-table use only.
    This is descriptive, not causal.
    """
    unique_vals = np.unique(feature_values)

    if len(unique_vals) <= 2:
        mask_one = feature_values == unique_vals.max()
        mask_zero = feature_values == unique_vals.min()

        if mask_one.sum() == 0 or mask_zero.sum() == 0:
            return "ambiguous", np.nan

        diff = float(np.mean(shap_values[mask_one]) - np.mean(shap_values[mask_zero]))
        if diff > 0:
            return "positive", diff
        if diff < 0:
            return "negative", diff
        return "ambiguous", diff

    if (
        pd.Series(feature_values).nunique(dropna=True) <= 1
        or pd.Series(shap_values).nunique(dropna=True) <= 1
    ):
        rho = np.nan
    else:
        rho = spearmanr(feature_values, shap_values, nan_policy="omit").correlation
    
    if np.isnan(rho) or abs(rho) < 0.05:
        return "ambiguous", float(rho) if not np.isnan(rho) else np.nan
    
    return ("positive" if rho > 0 else "negative"), float(rho)


def build_n3_direction_table(shap_obj: Any, X_test: pd.DataFrame, output_dir: Path) -> pd.DataFrame:
    shap_values = get_shap_values_matrix(shap_obj)

    rows = []
    for j, feature in enumerate(X_test.columns):
        direction, metric = approximate_n3_direction(
            X_test.iloc[:, j].to_numpy(),
            shap_values[:, j],
        )
        rows.append(
            {
                "feature_name": feature,
                "n3_broad_direction": direction,
                "n3_direction_metric": metric,
            }
        )

    out = pd.DataFrame(rows)
    out.to_csv(output_dir / "n3_directional_association_table.csv", index=False)
    return out


def build_l4_n3_comparison(
    l4_table: pd.DataFrame,
    shap_table: pd.DataFrame,
    n3_direction_table: pd.DataFrame,
    output_dir: Path,
) -> pd.DataFrame:
    l4_top = l4_table.copy().head(TOP_K_COMPARISON)
    l4_top["l4_rank"] = np.arange(1, len(l4_top) + 1)
    l4_top["l4_direction"] = np.where(
        l4_top["coefficient"] > 0,
        "positive",
        np.where(l4_top["coefficient"] < 0, "negative", "zero"),
    )

    n3_top = shap_table.copy().head(TOP_K_COMPARISON)
    n3_top["n3_rank"] = np.arange(1, len(n3_top) + 1)
    n3_top = n3_top.merge(n3_direction_table, on="feature_name", how="left")

    union_features = sorted(set(l4_top["feature_name"]) | set(n3_top["feature_name"]))
    base = pd.DataFrame({"feature_name": union_features})

    comparison = (
        base.merge(
            l4_top[
                [
                    "feature_name",
                    "display_name",
                    "block",
                    "l4_rank",
                    "standardized_effect",
                    "l4_direction",
                    "sign_stability",
                ]
            ],
            on="feature_name",
            how="left",
        )
        .merge(
            n3_top[
                [
                    "feature_name",
                    "display_name",
                    "block",
                    "n3_rank",
                    "mean_abs_shap",
                    "n3_broad_direction",
                ]
            ],
            on="feature_name",
            how="left",
            suffixes=("_l4", "_n3"),
        )
    )

    comparison["display_name"] = comparison["display_name_l4"].combine_first(comparison["display_name_n3"])
    comparison["block"] = comparison["block_l4"].combine_first(comparison["block_n3"])
    comparison["in_l4_top10"] = comparison["l4_rank"].notna()
    comparison["in_n3_top10"] = comparison["n3_rank"].notna()

    def direction_agreement(row: pd.Series) -> str:
        if not (row["in_l4_top10"] and row["in_n3_top10"]):
            return "not_applicable"
        if row["n3_broad_direction"] == "ambiguous" or row["l4_direction"] == "zero":
            return "ambiguous"
        return "agree" if row["l4_direction"] == row["n3_broad_direction"] else "disagree"

    comparison["directional_agreement"] = comparison.apply(direction_agreement, axis=1)

    comparison = comparison[
        [
            "feature_name",
            "display_name",
            "block",
            "in_l4_top10",
            "l4_rank",
            "standardized_effect",
            "l4_direction",
            "sign_stability",
            "in_n3_top10",
            "n3_rank",
            "mean_abs_shap",
            "n3_broad_direction",
            "directional_agreement",
        ]
    ].sort_values(
        ["in_l4_top10", "l4_rank", "in_n3_top10", "n3_rank"],
        ascending=[False, True, False, True],
    )

    comparison.to_csv(output_dir / "l4_vs_n3_top10_comparison.csv", index=False)
    return comparison

In [27]:
# Main execution
# =========================================================
# 13. MANIFEST
# =========================================================
def save_run_manifest(selected_dependence: pd.DataFrame, output_dir: Path) -> None:
    manifest = {
        "data_path": str(PANEL_PATH),
        "train_years": [2010, 2021],
        "test_years": [2022, 2024],
        "target": TARGET_COL,
        "seed": SEED,
        "permutation_repeats": PERMUTATION_REPEATS,
        "calibration_bins": CALIBRATION_BINS,
        "selected_dependence_features": selected_dependence["feature_name"].tolist(),
        "selected_dependence_blocks": selected_dependence["block"].tolist(),
        "selection_note": (
            "Top three substantive SHAP features on the 2022-2024 holdout, "
            "excluding helper/coverage indicators and FF17 dummies, "
            "with feature-block diversity enforced where possible."
        ),
        "l4_params": L4_PARAMS,
        "n3_params": N3_PARAMS,
        "package_versions_file": "package_versions.csv",
    }
    with open(output_dir / "run_manifest.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)


# =========================================================
# 14. MAIN
# =========================================================
def main() -> None:
    warnings.filterwarnings("ignore", category=FutureWarning)
    warnings.filterwarnings("ignore", category=UserWarning)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print("Checking shap/xgboost versions...")
    check_versions(OUTPUT_DIR)

    print("Loading panel...")
    df = load_panel()
    transformed_meta = build_transformed_metadata()

    train_df, test_df = split_panel(df)
    y_test = test_df[TARGET_COL].astype(int).to_numpy()

    print(
        f"Train rows (2010-2021): {len(train_df):,} | "
        f"Test rows (2022-2024): {len(test_df):,} | "
        f"Test positive rate: {y_test.mean():.4f}"
    )

    # -------------------------
    # L4
    # -------------------------
    print("\nFitting L4 on 2010-2021...")
    l4_model, _, X_train_l4, X_test_l4, l4_probs = fit_l4(train_df, test_df)
    l4_scores = evaluate_predictions(y_test, l4_probs, "L4_refit_2010_2021_scored_on_2022_2024")
    pd.DataFrame([l4_scores]).to_csv(OUTPUT_DIR / "l4_holdout_scores.csv", index=False)

    print("Computing L4 sign stability across six expanding folds...")
    l4_sign_stability = compute_l4_sign_stability(l4_model, train_df)

    print("Building L4 coefficient / odds ratio / standardized effect tables...")
    l4_appendix, l4_main = build_l4_tables(
        model=l4_model,
        X_train_transformed=X_train_l4,
        transformed_meta=transformed_meta,
        sign_stability=l4_sign_stability,
        output_dir=OUTPUT_DIR,
    )

    # -------------------------
    # N3
    # -------------------------
    print("\nFitting updated N3 on 2010-2021...")
    n3_model, _, X_train_n3, X_test_n3, n3_probs = fit_n3(train_df, test_df)
    n3_scores = evaluate_predictions(y_test, n3_probs, "N3_refit_2010_2021_scored_on_2022_2024")
    pd.DataFrame([n3_scores]).to_csv(OUTPUT_DIR / "n3_holdout_scores.csv", index=False)

    pd.DataFrame([l4_scores, n3_scores]).to_csv(OUTPUT_DIR / "t1_holdout_scores.csv", index=False)

    print("Computing N3 permutation importance on 2022-2024 holdout...")
    perm_df = permutation_importance_pr_auc(
        model=n3_model,
        X_test=X_test_n3,
        y_test=y_test,
        repeats=PERMUTATION_REPEATS,
        seed=SEED,
    )
    perm_df = perm_df.merge(transformed_meta, on="feature_name", how="left")
    perm_df.to_csv(OUTPUT_DIR / "n3_permutation_importance.csv", index=False)
    plot_permutation_importance(perm_df, OUTPUT_DIR)

    print("Computing N3 SHAP on full holdout...")
    shap_obj = compute_shap_outputs(
        model=n3_model,
        X_test=X_test_n3,
        test_df=test_df,
        probs=n3_probs,
        output_dir=OUTPUT_DIR,
    )

    print("Building N3 SHAP tables and summary plots...")
    shap_table = build_shap_tables(
        shap_obj=shap_obj,
        X_test=X_test_n3,
        transformed_meta=transformed_meta,
        output_dir=OUTPUT_DIR,
    )
    plot_shap_summary(shap_obj, X_test_n3, OUTPUT_DIR)

    print("Selecting and plotting three SHAP dependence plots...")
    selected_features, selected_df = choose_dependence_features(shap_table, OUTPUT_DIR)
    plot_shap_dependence(shap_obj, X_test_n3, selected_features, OUTPUT_DIR)

    print("Building N3 calibration outputs...")
    build_calibration_outputs(y_test=y_test, probs=n3_probs, output_dir=OUTPUT_DIR)

    print("Building L4 vs N3 comparison table...")
    n3_direction_table = build_n3_direction_table(shap_obj, X_test_n3, OUTPUT_DIR)
    build_l4_n3_comparison(
        l4_table=l4_appendix,
        shap_table=shap_table,
        n3_direction_table=n3_direction_table,
        output_dir=OUTPUT_DIR,
    )

    save_run_manifest(selected_df, OUTPUT_DIR)

    print(f"\nT1 interpretability complete. Outputs written to: {OUTPUT_DIR.resolve()}")

# Run in Jupyter or as a normal .py file
main()

Checking shap/xgboost versions...

Package versions:
package version
   shap  0.49.1
xgboost   2.1.4

Loading panel...
Train rows (2010-2021): 47,793 | Test rows (2022-2024): 13,147 | Test positive rate: 0.0655

Fitting L4 on 2010-2021...
Computing L4 sign stability across six expanding folds...
Building L4 coefficient / odds ratio / standardized effect tables...

Fitting updated N3 on 2010-2021...
Computing N3 permutation importance on 2022-2024 holdout...
Computing N3 SHAP on full holdout...
Building N3 SHAP tables and summary plots...
Selecting and plotting three SHAP dependence plots...
Building N3 calibration outputs...
Building L4 vs N3 comparison table...

T1 interpretability complete. Outputs written to: /home/jovyan/work/Senior Thesis/1. Final Report /Model Runs/t1_interpretability_outputs


In [26]:
import json
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("t1_interpretability_outputs")

final_features = [
    "prior_campaign_count_5y",
    "market_cap",
    "log_tobins_q_rel_peer",
]

final_blocks = [
    "Activism history",
    "Market",
    "Peer-relative",
]

selection_note = (
    "Final thesis main-text dependence plots: prior_campaign_count_5y, "
    "market_cap, and log_tobins_q_rel_peer. log_tobins_q_rel_peer replaced "
    "current_ratio as the third plot because it provided a cleaner and more "
    "interpretable main-text figure while preserving feature-block diversity."
)

# 1) Update run_manifest.json if it exists
manifest_path = OUTPUT_DIR / "run_manifest.json"
if manifest_path.exists():
    with open(manifest_path, "r") as f:
        manifest = json.load(f)

    manifest["selected_dependence_features"] = final_features
    manifest["selected_dependence_blocks"] = final_blocks
    manifest["selection_note"] = selection_note

    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)

    print(f"Updated: {manifest_path}")
else:
    print(f"Not found: {manifest_path}")

# 2) Overwrite n3_selected_dependence_features.csv with the final trio
selected_csv_path = OUTPUT_DIR / "n3_selected_dependence_features.csv"
selected_df = pd.DataFrame(
    {
        "rank": [1, 2, 3],
        "feature": final_features,
        "block": final_blocks,
        "main_text_final": [True, True, True],
    }
)
selected_df.to_csv(selected_csv_path, index=False)
print(f"Updated: {selected_csv_path}")

# 3) Save a tiny note for yourself if useful
notes_path = OUTPUT_DIR / "dependence_plot_final_choice_note.txt"
with open(notes_path, "w") as f:
    f.write(selection_note + "\n")
print(f"Saved note: {notes_path}")

Updated: t1_interpretability_outputs/run_manifest.json
Updated: t1_interpretability_outputs/n3_selected_dependence_features.csv
Saved note: t1_interpretability_outputs/dependence_plot_final_choice_note.txt
